# TIC Meta-Test Results Analysis

This notebook analyzes results from meta test scripts (120, 220, 320).

Set `test_prefix` below to select which test results to analyze.
For test 320 (dual device), also set `device` to "d1" or "d2".

In [ ]:
# === SELECT TEST PREFIX HERE ===
test_prefix = "320"  # Options: "120", "220", "320"
device = "d2"        # For 320 only: "d1" or "d2"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import glob

# Find most recent results for selected prefix
if test_prefix == "320":
    pattern = f'logs/{test_prefix}_*/results_{device}.csv'
else:
    pattern = f'logs/{test_prefix}_*/results.csv'

log_dirs = sorted(glob.glob(pattern))
if log_dirs:
    results_file = log_dirs[-1]
    print(f'Test prefix: {test_prefix}')
    if test_prefix == "320":
        print(f'Device: {device}')
    print(f'Loading: {results_file}')
else:
    print(f'No results found for prefix {test_prefix}')
    results_file = input('Enter path to results.csv: ')

df = pd.read_csv(results_file)

# Calculate delay error from measured - configured
df['delay_error_ns'] = df['delay_measured_ns'] - df['delay_configured_ns']

print(f'Loaded {len(df)} test results')
df.head()

## Summary Statistics

In [ ]:
# Filter to successful tests
passed = df[df['status'] == 'PASS']
failed = df[df['status'] == 'FAIL']

print(f'Total tests: {len(df)}')
print(f'Passed: {len(passed)} ({100*len(passed)/len(df):.1f}%)')
print(f'Failed: {len(failed)} ({100*len(failed)/len(df):.1f}%)')

if len(passed) > 0:
    print(f'\nDelay Error Statistics (passed tests):')
    print(f'  Min:    {passed["delay_error_ns"].min():.2f} ns')
    print(f'  Max:    {passed["delay_error_ns"].max():.2f} ns')
    print(f'  Mean:   {passed["delay_error_ns"].mean():.2f} ns')
    print(f'  StdDev: {passed["delay_error_ns"].std():.2f} ns')
    print(f'  Median: {passed["delay_error_ns"].median():.2f} ns')

## Delay Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of delay errors
ax = axes[0]
if len(passed) > 0:
    ax.hist(passed['delay_error_ns'], bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--', label='Zero error')
    ax.axvline(x=12.5, color='g', linestyle='--', label='+12.5ns tolerance')
    ax.axvline(x=-12.5, color='g', linestyle='--', label='-12.5ns tolerance')
    ax.set_xlabel('Delay Error (ns)')
    ax.set_ylabel('Count')
    ax.set_title('Delay Error Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Box plot
ax = axes[1]
if len(passed) > 0:
    ax.boxplot(passed['delay_error_ns'], vert=True)
    ax.axhline(y=0, color='r', linestyle='--')
    ax.axhline(y=12.5, color='g', linestyle='--')
    ax.axhline(y=-12.5, color='g', linestyle='--')
    ax.set_ylabel('Delay Error (ns)')
    ax.set_title('Delay Error Box Plot')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Delay Error vs Parameters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Delay error vs configured delay
ax = axes[0]
if len(passed) > 0:
    ax.scatter(passed['delay_configured_ns'], passed['delay_error_ns'], alpha=0.6)
    ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=12.5, color='g', linestyle='--', alpha=0.5)
    ax.axhline(y=-12.5, color='g', linestyle='--', alpha=0.5)
    ax.set_xlabel('Configured Delay (ns)')
    ax.set_ylabel('Delay Error (ns)')
    ax.set_title('Delay Error vs Configured Delay')
    ax.grid(True, alpha=0.3)

# Delay error vs frequency
ax = axes[1]
if len(passed) > 0:
    ax.scatter(passed['freq_configured_hz']/1000, passed['delay_error_ns'], alpha=0.6)
    ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=12.5, color='g', linestyle='--', alpha=0.5)
    ax.axhline(y=-12.5, color='g', linestyle='--', alpha=0.5)
    ax.set_xlabel('Configured Frequency (kHz)')
    ax.set_ylabel('Delay Error (ns)')
    ax.set_title('Delay Error vs Frequency')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Measured vs Configured Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Measured vs configured delay
ax = axes[0]
if len(passed) > 0:
    max_delay = max(passed['delay_configured_ns'].max(), passed['delay_measured_ns'].max())
    ax.scatter(passed['delay_configured_ns'], passed['delay_measured_ns'], alpha=0.6)
    ax.plot([0, max_delay], [0, max_delay], 'r--', label='Perfect match')
    ax.set_xlabel('Configured Delay (ns)')
    ax.set_ylabel('Measured Delay (ns)')
    ax.set_title('Measured vs Configured Delay')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')

# Frequency error
ax = axes[1]
if len(passed) > 0:
    freq_error_pct = 100 * (passed['freq_a_measured_hz'] - passed['freq_configured_hz']) / passed['freq_configured_hz']
    ax.hist(freq_error_pct, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Frequency Error (%)')
    ax.set_ylabel('Count')
    ax.set_title('Frequency Measurement Error Distribution')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Failure Analysis (if any)

In [ ]:
if len(failed) > 0:
    print('Failed tests:')
    # Select columns that exist
    cols = ['test_num', 'freq_configured_hz', 'delay_configured_ns', 
            'delay_measured_ns', 'delay_error_ns']
    if 'fail_stage' in failed.columns:
        cols.append('fail_stage')
    display(failed[cols])
    
    # Analyze failure patterns
    fig, ax = plt.subplots(figsize=(10, 6))
    period_ns = 1e9 / failed['freq_configured_hz']
    delay_ratio = failed['delay_configured_ns'] / period_ns
    ax.scatter(delay_ratio, failed['delay_error_ns'])
    ax.set_xlabel('Delay / Period Ratio')
    ax.set_ylabel('Delay Error (ns)')
    ax.set_title('Failed Tests: Delay Error vs Delay/Period Ratio')
    ax.axvline(x=0.25, color='r', linestyle='--', label='Quarter period limit')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print('No failed tests!')

## Summary

In [ ]:
if len(passed) > 0:
    print('='*50)
    print('TEST SUMMARY')
    print('='*50)
    print(f'Pass rate: {100*len(passed)/len(df):.1f}%')
    print(f'\nDelay accuracy:')
    print(f'  Mean error: {passed["delay_error_ns"].mean():.2f} ns')
    print(f'  Max |error|: {passed["delay_error_ns"].abs().max():.2f} ns')
    print(f'  Within 12.5ns: {100*sum(passed["delay_error_ns"].abs() <= 12.5)/len(passed):.1f}%')
    print(f'\nFrequency range tested: {df["freq_configured_hz"].min()/1000:.1f} - {df["freq_configured_hz"].max()/1000:.1f} kHz')
    print(f'Delay range tested: {df["delay_configured_ns"].min():.0f} - {df["delay_configured_ns"].max():.0f} ns')